In [1]:
import pandas as pd
import requests

In [2]:
url = 'https://remoteok.com/api?tag=data'
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        ' (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
}

In [3]:
response = requests.get(url, headers=headers)

In [4]:
if response.status_code == 200:
    data = response.json()
    raw_jobs_data = []

    for job in data[1:]:
        raw_jobs_data.append({
            'Job Title': job.get('position', 'N/A'),
            'Company': job.get('company', 'N/A'),
            'Location': job.get('location', 'Worldwide'),
            'Skills / Tech Stack': ', '.join(job.get('tags', []))
            if job.get('tags')
            else 'Not specified',
            'Post Date': job.get('date', 'N/A'),
        })
        
    df_raw = pd.DataFrame(raw_jobs_data)
    print('Məlumatlar saytdan çəkilir...')
    print(
        f'{len(df_raw)} xam məlumat toplandı.\n'
    )
    display(df_raw)
else:
    print('Sorğuda xəta baş verdi:', response.status_code)

Məlumatlar saytdan çəkilir...
100 xam məlumat toplandı.



,Job Title,Company,Location,Skills / Tech Stack,Post Date
0,Stoic,New Atlantis,"Well,","teaching, data annotation",2026-08-06T11:15:44+00:00
1,Cabin Crew Virtual Interviews PAN India,Akasa Air,"Bengaluru,","saas, customer support, marketing, exec, ops, ...",2026-08-05T11:45:49+00:00
2,Brand Strategy,Capgemini,"Gurgaon,","saas, customer support, marketing, exec, ops, ...",2026-08-05T10:55:32+00:00
3,Handyperson,Hilton,"Surfers Paradise,","customer support, marketing, travel, speech, f...",2026-08-05T09:19:25+00:00
4,The quest build a better AI tutor,Cameron Newspapers | Citizen Observer &amp; Ca...,"Cameron,","teaching, data annotation",2026-08-05T02:38:35+00:00
...,...,...,...,...,...
95,Data Science,Capgemini,"Gurgaon,","sys admin, marketing, travel, exec, cloud, sal...",2026-07-07T10:54:25+00:00
96,eLearning Developer,The Learning Network,"London, London, Ontario, Canada","writer, customer support, exec, finance, micro...",2026-07-07T00:00:00+00:00
97,Lead Data Scientist,Brigit,San Francisco,"python, education, customer support, testing, ...",2026-07-04T16:00:08+00:00
98,Administrative Data Entry File Clerk,Recruitlytics Hiring,"Alberta, Alberta, Canada","design, saas, embedded, customer support, mark...",2026-07-04T08:07:12+00:00


In [5]:
print(' Xam Məlumatın Yoxlanılması:\n')

amp_count = df_raw['Job Title'].str.contains('&amp;').sum()
print(f" '&amp;' simvolu olan sətir sayı: {amp_count}")

sample_date = df_raw['Post Date'].iloc[0]
print(f' Nümunə tarix formatı: {sample_date}')

print('\n Uyğunsuz Simvolların Nümunəsi')
display(df_raw[['Job Title', 'Company', 'Location']].head(5))

 Xam Məlumatın Yoxlanılması:

 '&amp;' simvolu olan sətir sayı: 2
 Nümunə tarix formatı: 2026-08-06T11:15:44+00:00

 Uyğunsuz Simvolların Nümunəsi


,Job Title,Company,Location
0,Stoic,New Atlantis,"Well,"
1,Cabin Crew Virtual Interviews PAN India,Akasa Air,"Bengaluru,"
2,Brand Strategy,Capgemini,"Gurgaon,"
3,Handyperson,Hilton,"Surfers Paradise,"
4,The quest build a better AI tutor,Cameron Newspapers | Citizen Observer &amp; Ca...,"Cameron,"


In [6]:
import html
df_clean = df_raw.copy()

df_clean['Job Title'] = df_clean['Job Title'].apply(html.unescape)
df_clean['Company'] = df_clean['Company'].apply(html.unescape)

def clean_location(loc):
    if loc and not str(loc).isascii():
        try:
            return str(loc).encode('latin1').decode('utf-8')
        except Exception:
            return 'Worldwide'
    return loc if loc else 'Worldwide'

df_clean['Location'] = df_clean['Location'].apply(clean_location)

df_clean['Post Date'] = (
    df_clean['Post Date'].astype(str).str.split('T').str[0]
)

df_clean.to_csv('remote_job_market_data.csv', index=False, encoding='utf-8-sig')


In [7]:
print('Məlumatlar tam təmizləndi və remote_job_market_data.csv faylına yazıldı.\n')
display(df_clean)

Məlumatlar tam təmizləndi və remote_job_market_data.csv faylına yazıldı.



,Job Title,Company,Location,Skills / Tech Stack,Post Date
0,Stoic,New Atlantis,"Well,","teaching, data annotation",2026-08-06
1,Cabin Crew Virtual Interviews PAN India,Akasa Air,"Bengaluru,","saas, customer support, marketing, exec, ops, ...",2026-08-05
2,Brand Strategy,Capgemini,"Gurgaon,","saas, customer support, marketing, exec, ops, ...",2026-08-05
3,Handyperson,Hilton,"Surfers Paradise,","customer support, marketing, travel, speech, f...",2026-08-05
4,The quest build a better AI tutor,Cameron Newspapers | Citizen Observer & Camero...,"Cameron,","teaching, data annotation",2026-08-05
...,...,...,...,...,...
95,Data Science,Capgemini,"Gurgaon,","sys admin, marketing, travel, exec, cloud, sal...",2026-07-07
96,eLearning Developer,The Learning Network,"London, London, Ontario, Canada","writer, customer support, exec, finance, micro...",2026-07-07
97,Lead Data Scientist,Brigit,San Francisco,"python, education, customer support, testing, ...",2026-07-04
98,Administrative Data Entry File Clerk,Recruitlytics Hiring,"Alberta, Alberta, Canada","design, saas, embedded, customer support, mark...",2026-07-04
